# 🔄 Notebook 2: Simple Polling

Simple polling is the most straightforward approach to getting updates from a server. It's not truly "real-time," but it's a great baseline and works for many use cases!

## Learning Objectives

By the end of this notebook, you'll understand:
- How simple polling works
- When to use it (and when not to)
- How to implement a polling client and server
- Trade-offs and optimizations

## 🤔 What is Simple Polling?

Simple polling is exactly what it sounds like: the client repeatedly asks the server "Do you have anything new for me?"

```
┌────────┐                              ┌────────┐
│ Client │                              │ Server │
└───┬────┘                              └───┬────┘
    │                                       │
    │──── GET /updates ────────────────────►│
    │◄─── {"updates": []} ─────────────────│  (nothing new)
    │                                       │
    │     ⏳ Wait 2 seconds...              │
    │                                       │
    │──── GET /updates ────────────────────►│
    │◄─── {"updates": []} ─────────────────│  (still nothing)
    │                                       │
    │     ⏳ Wait 2 seconds...              │
    │                                       │
    │──── GET /updates ────────────────────►│
    │◄─── {"updates": [{...}]} ────────────│  (got something!)
    │                                       │
```

It's like checking your mailbox every hour. Simple, but you might miss time-sensitive letters!

## 🛠️ Let's Build It!

We'll create a simple chat room where messages are fetched via polling.

### Step 1: Start the Server

Before continuing, start the server in a terminal:

```bash
cd 04-patterns/real-time-updates/servers
python simple_polling_server.py
```

You should see: `🚀 Starting Simple Polling Server on port 5001`

In [1]:
import requests

try:
    response = requests.get("http://localhost:5001/health", timeout=2)
    if response.status_code == 200:
        print("✅ Server is running!")
    else:
        print("⚠️ Server responded but with unexpected status")
except requests.exceptions.ConnectionError:
    print("❌ Server is not running!")
    print("   Please start it with: python ../servers/simple_polling_server.py")

✅ Server is running!


### Step 2: Create the Polling Client

In [2]:
import requests
import time
from datetime import datetime

class SimplePollingClient:
    """
    A simple polling client that checks for new messages at regular intervals.
    """
    
    def __init__(self, server_url: str, poll_interval: float = 2.0):
        self.server_url = server_url
        self.poll_interval = poll_interval
        self.last_timestamp = 0  # Track when we last got messages
        self.running = False
    
    def poll_once(self):
        """
        Make a single poll request to the server.
        Returns new messages since last poll.
        """
        try:
            response = requests.get(
                f"{self.server_url}/messages",
                params={"since": self.last_timestamp},
                timeout=5
            )
            
            if response.status_code == 200:
                data = response.json()
                messages = data.get("messages", [])
                
                # Update our timestamp to the latest message
                if messages:
                    self.last_timestamp = max(msg["timestamp"] for msg in messages)
                
                return messages
            else:
                print(f"⚠️ Server returned status {response.status_code}")
                return []
                
        except requests.exceptions.RequestException as e:
            print(f"❌ Request failed: {e}")
            return []
    
    def send_message(self, user: str, text: str):
        """
        Send a message to the chat.
        """
        try:
            response = requests.post(
                f"{self.server_url}/messages",
                json={"user": user, "text": text},
                timeout=5
            )
            return response.status_code == 201
        except requests.exceptions.RequestException as e:
            print(f"❌ Failed to send message: {e}")
            return False

# Create client instance
client = SimplePollingClient("http://localhost:5001")
print("✅ Polling client created!")

✅ Polling client created!


In [3]:
# Let's send some test messages!

print("📤 Sending test messages...\n")

client.send_message("Alice", "Hello everyone!")
time.sleep(0.5)
client.send_message("Bob", "Hey Alice! How are you?")
time.sleep(0.5)
client.send_message("Alice", "Doing great! Just learning about polling.")

print("\n✅ Messages sent!")

📤 Sending test messages...




✅ Messages sent!


In [4]:
# Now let's poll for messages!

print("🔄 Polling for messages...\n")

messages = client.poll_once()

if messages:
    print(f"📬 Received {len(messages)} message(s):\n")
    for msg in messages:
        timestamp = datetime.fromtimestamp(msg['timestamp']).strftime('%H:%M:%S')
        print(f"  [{timestamp}] {msg['user']}: {msg['text']}")
else:
    print("📭 No new messages")

🔄 Polling for messages...

📬 Received 7 message(s):

  [11:06:12] Alice: Hello everyone!
  [11:06:12] Bob: Hey Alice! How are you?
  [11:06:13] Alice: Doing great! Just learning about polling.
  [11:06:18] Charlie: I just joined!
  [11:19:08] Alice: Hello everyone!
  [11:19:08] Bob: Hey Alice! How are you?
  [11:19:09] Alice: Doing great! Just learning about polling.


In [5]:
# Let's see continuous polling in action!
# We'll poll for 10 seconds and see what happens

import threading

def continuous_poll(client, duration=10, interval=2):
    """
    Poll continuously for a set duration.
    """
    print(f"🔄 Starting continuous polling for {duration} seconds...")
    print(f"   Polling every {interval} seconds\n")
    
    start_time = time.time()
    poll_count = 0
    
    while time.time() - start_time < duration:
        poll_count += 1
        current_time = datetime.now().strftime('%H:%M:%S')
        
        messages = client.poll_once()
        
        if messages:
            print(f"[{current_time}] Poll #{poll_count}: 📬 {len(messages)} new message(s)")
            for msg in messages:
                print(f"            └─ {msg['user']}: {msg['text']}")
        else:
            print(f"[{current_time}] Poll #{poll_count}: 📭 No new messages")
        
        time.sleep(interval)
    
    print(f"\n✅ Polling complete! Made {poll_count} requests.")

# Reset client timestamp to see all messages
client.last_timestamp = 0

# Start polling in background
poll_thread = threading.Thread(target=continuous_poll, args=(client, 10, 2))
poll_thread.start()

# Wait a bit then send a new message
time.sleep(5)
print("\n📤 Sending a new message during polling...\n")
client.send_message("Charlie", "I just joined!")

# Wait for polling to complete
poll_thread.join()

🔄 Starting continuous polling for 10 seconds...
   Polling every 2 seconds

[11:19:09] Poll #1: 📬 7 new message(s)
            └─ Alice: Hello everyone!
            └─ Bob: Hey Alice! How are you?
            └─ Alice: Doing great! Just learning about polling.
            └─ Charlie: I just joined!
            └─ Alice: Hello everyone!
            └─ Bob: Hey Alice! How are you?
            └─ Alice: Doing great! Just learning about polling.


[11:19:11] Poll #2: 📭 No new messages


[11:19:13] Poll #3: 📭 No new messages



📤 Sending a new message during polling...



[11:19:15] Poll #4: 📬 1 new message(s)
            └─ Charlie: I just joined!


[11:19:17] Poll #5: 📭 No new messages



✅ Polling complete! Made 5 requests.


## 📊 Understanding the Trade-offs

Let's visualize what happens with different polling intervals:

In [6]:
def analyze_polling_tradeoffs():
    """
    Analyze the trade-offs of different polling intervals.
    """
    print("📊 Polling Interval Trade-offs")
    print("="*60)
    print("")
    
    # Calculate requests per hour for different intervals
    intervals = [
        (0.5, "Very fast (500ms)"),
        (2, "Fast (2s)"),
        (5, "Medium (5s)"),
        (30, "Slow (30s)"),
        (60, "Very slow (1min)")
    ]
    
    print(f"{'Interval':<25} {'Requests/Hour':<15} {'Max Latency':<15}")
    print("-"*55)
    
    for interval, name in intervals:
        requests_per_hour = 3600 / interval
        max_latency = f"{interval}s"
        print(f"{name:<25} {requests_per_hour:<15.0f} {max_latency:<15}")
    
    print("")
    print("💡 Key Insight:")
    print("   - Faster polling = More requests = Higher server load")
    print("   - Slower polling = Fewer requests = Higher latency")
    print("")
    print("📈 With 10,000 users:")
    print("   - 500ms polling: 72 million requests/hour!")
    print("   - 30s polling: 1.2 million requests/hour")

analyze_polling_tradeoffs()

📊 Polling Interval Trade-offs

Interval                  Requests/Hour   Max Latency    
-------------------------------------------------------
Very fast (500ms)         7200            0.5s           
Fast (2s)                 1800            2s             
Medium (5s)               720             5s             
Slow (30s)                120             30s            
Very slow (1min)          60              60s            

💡 Key Insight:
   - Faster polling = More requests = Higher server load
   - Slower polling = Fewer requests = Higher latency

📈 With 10,000 users:
   - 500ms polling: 72 million requests/hour!
   - 30s polling: 1.2 million requests/hour


## ✅ Advantages of Simple Polling

1. **Dead simple to implement** - Just HTTP requests!
2. **Stateless** - No connection to maintain
3. **Works everywhere** - Any HTTP client works
4. **Easy to debug** - Standard request/response
5. **No special infrastructure** - Works with any load balancer

## ❌ Disadvantages

1. **Higher latency** - Updates delayed by polling interval
2. **Wasted requests** - Most polls return empty
3. **Inefficient** - Server load scales with users × poll rate
4. **Not truly real-time** - Inherent delay

In [7]:
# Let's measure the "wasted" requests problem

def measure_polling_efficiency(poll_count=10, interval=1):
    """
    Measure how many polls return empty vs with data.
    """
    # Reset client
    client.last_timestamp = time.time()
    
    empty_polls = 0
    useful_polls = 0
    
    print(f"🔬 Running {poll_count} polls with {interval}s interval...\n")
    
    for i in range(poll_count):
        messages = client.poll_once()
        if messages:
            useful_polls += 1
            print(f"  Poll {i+1}: 📬 Got {len(messages)} message(s)")
        else:
            empty_polls += 1
            print(f"  Poll {i+1}: 📭 Empty")
        time.sleep(interval)
    
    efficiency = (useful_polls / poll_count) * 100
    
    print(f"\n📊 Results:")
    print(f"   Empty polls: {empty_polls} ({100-efficiency:.1f}%)")
    print(f"   Useful polls: {useful_polls} ({efficiency:.1f}%)")
    print(f"\n💡 This is typical! Most polls return nothing.")

measure_polling_efficiency()

🔬 Running 10 polls with 1s interval...

  Poll 1: 📭 Empty


  Poll 2: 📭 Empty


  Poll 3: 📭 Empty


  Poll 4: 📭 Empty


  Poll 5: 📭 Empty


...

  Poll 7: 📭 Empty


  Poll 8: 📭 Empty


  Poll 9: 📭 Empty


  Poll 10: 📭 Empty



📊 Results:
   Empty polls: 10 (100.0%)
   Useful polls: 0 (0.0%)

💡 This is typical! Most polls return nothing.


## 🎯 When to Use Simple Polling

Simple polling is a great choice when:

| Use Case | Why Polling Works |
|----------|-------------------|
| Dashboard updates | 5-10s delay is acceptable |
| Email inbox | Users don't expect instant updates |
| Social media feeds | "Pull to refresh" is expected |
| Status pages | Updates are infrequent |
| Analytics | Near real-time is good enough |

### Don't use it when:

- Users expect **instant** updates (chat, gaming)
- Updates happen **very frequently** (stock tickers)
- You have **millions of users** (server load issue)
- Latency is **critical** (live auctions)

## 🔧 Optimization: HTTP Keep-Alive

One way to reduce polling overhead is using HTTP keep-alive connections. This avoids the TCP handshake for each request.

In [8]:
import time
import requests

def compare_keep_alive():
    """
    Compare request times with and without keep-alive.
    """
    url = "http://localhost:5001/messages?since=0"
    
    # Without keep-alive (new connection each time)
    print("🔄 Without Keep-Alive (new TCP connection each request):")
    times_no_keepalive = []
    for i in range(5):
        start = time.time()
        requests.get(url)
        elapsed = (time.time() - start) * 1000
        times_no_keepalive.append(elapsed)
        print(f"   Request {i+1}: {elapsed:.2f}ms")
    avg_no_ka = sum(times_no_keepalive) / len(times_no_keepalive)
    print(f"   Average: {avg_no_ka:.2f}ms\n")
    
    # With keep-alive (reuse connection)
    print("🔗 With Keep-Alive (reuse TCP connection):")
    times_keepalive = []
    session = requests.Session()  # Sessions use keep-alive by default
    for i in range(5):
        start = time.time()
        session.get(url)
        elapsed = (time.time() - start) * 1000
        times_keepalive.append(elapsed)
        print(f"   Request {i+1}: {elapsed:.2f}ms")
    avg_ka = sum(times_keepalive) / len(times_keepalive)
    print(f"   Average: {avg_ka:.2f}ms")
    
    if avg_ka < avg_no_ka:
        improvement = ((avg_no_ka - avg_ka) / avg_no_ka) * 100
        print(f"\n✅ Keep-alive is {improvement:.1f}% faster!")

compare_keep_alive()

🔄 Without Keep-Alive (new TCP connection each request):
   Request 1: 3.96ms
   Request 2: 2.64ms
   Request 3: 2.65ms
   Request 4: 2.13ms
   Request 5: 2.06ms
   Average: 2.69ms

🔗 With Keep-Alive (reuse TCP connection):
   Request 1: 1.53ms
   Request 2: 1.23ms
   Request 3: 1.05ms
   Request 4: 0.87ms
   Request 5: 0.82ms
   Average: 1.10ms

✅ Keep-alive is 59.1% faster!


## 🧪 Quick Quiz

1. **You're building a weather dashboard that updates every 5 minutes. Should you use simple polling?**

2. **Your polling interval is 2 seconds. What's the maximum delay a user might experience for seeing a new message?**

3. **You have 100,000 users polling every 5 seconds. How many requests per second is that?**

In [9]:
# Run this to see the answers!

print("📝 Quiz Answers")
print("="*50)
print("")
print("1. YES! Weather updates every 5 minutes is perfect for polling.")
print("   You could poll every 1-2 minutes and still be very responsive.")
print("")
print("2. Maximum delay = 2 seconds (the polling interval) + network latency")
print("   If a message arrives right after a poll, user waits the full interval.")
print("")
print("3. 100,000 users ÷ 5 seconds = 20,000 requests per second!")
print("   This is why polling doesn't scale well for large user bases.")

📝 Quiz Answers

1. YES! Weather updates every 5 minutes is perfect for polling.
   You could poll every 1-2 minutes and still be very responsive.

2. Maximum delay = 2 seconds (the polling interval) + network latency
   If a message arrives right after a poll, user waits the full interval.

3. 100,000 users ÷ 5 seconds = 20,000 requests per second!
   This is why polling doesn't scale well for large user bases.


## 📚 Summary

### What We Learned:

1. **Simple polling** = Client repeatedly asks server for updates
2. **Easy to implement** but not truly real-time
3. **Trade-off**: Faster polling = more load, slower = more latency
4. **Most polls are wasted** - return no new data
5. **HTTP keep-alive** reduces overhead

### Interview Tips:

> "I'm going to start with a simple polling approach so I can focus on [core problem]. We can switch to something more sophisticated if we need lower latency."

This shows you understand trade-offs and can prioritize!

### Next Up: Long Polling

In the next notebook, we'll see how **long polling** improves on simple polling by holding requests open until there's new data.

In [10]:
# Cleanup: You can stop the server now if you want
print("🧹 Cleanup:")
print("   To stop the server, press Ctrl+C in the terminal where it's running.")
print("   Or keep it running for the next notebook!")

🧹 Cleanup:
   To stop the server, press Ctrl+C in the terminal where it's running.
   Or keep it running for the next notebook!
